# 02b · Feature selection on grouped training folds

Compare all transformed features with mutual information (MI) top-k selection and L1-based selection. All configurations use the same logistic-regression classifier and five accident-grouped folds. The primary metric is validation average precision (PR-AUC). The evaluation covers 2019–2023.

## Training data and grouped folds

Apply the year predicate in the PyArrow Parquet read. Reuse the Week 2 feature engineering and exclusion list. The shared full train/test split requires both year partitions, so use its exclusion list with the filtered training frame.

In [1]:
from pathlib import Path
from time import perf_counter
import sys
import warnings

try:
    import resource
except ImportError:
    resource = None

import numpy as np
import pandas as pd
from scipy.sparse import issparse
from sklearn.exceptions import ConvergenceWarning

root = Path.cwd()
if not (root / "src").is_dir():
    root = root.parent
sys.path.insert(0, str(root))

from src.config import GROUP_COL, PROCESSED_DIR, RANDOM_STATE, TARGET, TRAIN_YEARS
from src.evaluation.metrics import evaluate
from src.features.build_features import add_features
from src.features.preprocess import CATEGORICAL_FEATURES, NUMERIC_FEATURES
from src.features.select import MutualInformationPreprocessor, build_selector
from src.features.split import _NEVER_FEATURES, get_cv
from src.models.baseline import logistic_baseline

persons = pd.read_parquet(
    PROCESSED_DIR / "persons.parquet",
    engine="pyarrow",
    filters=[("year", "in", TRAIN_YEARS)],
)
assert not persons[["year", TARGET, GROUP_COL]].isna().any().any()
assert sorted(persons["year"].unique()) == TRAIN_YEARS
assert len(persons) == 619_552, (
    f"Expected 619,552 training rows, found {len(persons):,}. "
    "A mismatch may indicate a different dataset snapshot."
)
assert set(persons[TARGET].unique()) == {0, 1}
print(persons.groupby("year")[TARGET].agg(rows="size", ksi="sum", prevalence="mean").to_string())
featured = add_features(persons)
X = featured.drop(columns=list(_NEVER_FEATURES), errors="ignore")
y = featured[TARGET].to_numpy(dtype=np.int8)
groups = featured[GROUP_COL].to_numpy()
assert _NEVER_FEATURES.isdisjoint(X.columns)
assert (set(NUMERIC_FEATURES) | set(CATEGORICAL_FEATURES)) <= set(X.columns)
print(f"Training rows: {len(X):,}; accidents: {pd.Series(groups).nunique():,}; KSI: {y.sum():,} ({y.mean():.4%})")
print("Excluded columns present in source:", ", ".join(sorted(_NEVER_FEATURES & set(featured))))
print(f"Inputs after exclusions: {X.shape[1]}; columns passed to preprocessor: "
      f"{len(NUMERIC_FEATURES) + len(CATEGORICAL_FEATURES)}")
del persons, featured

        rows    ksi  prevalence
year                           
2019  132977  24356    0.183159
2020  105295  19555    0.185716
2021  129188  22312    0.172710
2022  126421  22810    0.180429
2023  125671  22669    0.180384
Training rows: 619,552; accidents: 273,226; KSI: 111,702 (18.0295%)
Excluded columns present in source: KSI, Num_Acc, hrmn, id_vehicule, jour, person_id, year
Inputs after exclusions: 48; columns passed to preprocessor: 41


In [2]:
start = perf_counter()
folds = list(get_cv().split(X, y, groups))
fold_seconds = perf_counter() - start
fold_rows = []
validation_hits = np.zeros(len(y), dtype=np.int8)
for fold, (train, valid) in enumerate(folds, start=1):
    train_groups = set(groups[train])
    valid_groups = set(groups[valid])
    overlap = len(train_groups & valid_groups)
    assert overlap == 0
    validation_hits[valid] += 1
    fold_rows.append({
        "fold": fold, "train_rows": len(train), "validation_rows": len(valid),
        "train_accidents": len(train_groups), "validation_accidents": len(valid_groups),
        "train_ksi_rate": y[train].mean(), "validation_ksi_rate": y[valid].mean(),
        "accident_overlap": overlap,
    })
assert len(folds) == 5 and (validation_hits == 1).all()
del train_groups, valid_groups, validation_hits
print(pd.DataFrame(fold_rows).to_string(index=False, float_format="{:.5f}".format))
print(f"Fold generation: {fold_seconds:.1f} s")

 fold  train_rows  validation_rows  train_accidents  validation_accidents  train_ksi_rate  validation_ksi_rate  accident_overlap
    1      495641           123911           218560                 54666         0.18029              0.18030                 0
    2      495642           123910           218667                 54559         0.18029              0.18030                 0
    3      495641           123911           218430                 54796         0.18030              0.18029                 0
    4      495643           123909           218650                 54576         0.18030              0.18029                 0
    5      495641           123911           218597                 54629         0.18030              0.18029                 0
Fold generation: 15.7 s


## Serial whole-accident pilot

Sample roughly 100,000 people by shuffling unique accident IDs inside the first training fold. Use all members of each selected accident. Time each stage of the pipeline separately. Peak RSS is the cumulative kernel resident-memory high-water mark, including data loading and fold generation. It is reported as unavailable on platforms without the resource module.

The helper fits the three stages in pipeline order on the supplied training indices. It captures warnings with their stage and prints them. Validation transforms and prediction occur after fitting.

In [3]:
def kernel_peak_rss_mib():
    if resource is None:
        return np.nan
    rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return rss / (1024**2 if sys.platform == "darwin" else 1024)


def format_kernel_peak_rss():
    peak = kernel_peak_rss_mib()
    return "unavailable" if np.isnan(peak) else f"{peak:.2f} MiB"


def timed_fit(stage, operation, timings, messages):
    start = perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        result = operation()
    timings[f"{stage}_s"] = perf_counter() - start
    for warning in caught:
        message = f"{stage}: {warning.category.__name__}: {warning.message}"
        messages.append(message)
        print(message, flush=True)
    return result


def fit_steps(train_X, train_y, method=None, **settings):
    baseline = logistic_baseline()
    prep = baseline.named_steps["prep"]
    model = baseline.named_steps["model"]
    if method == "mi":
        prep = MutualInformationPreprocessor([("columns", prep)])

    timings = {"selection_s": 0.0}
    messages = []
    matrix = timed_fit("prep", lambda: prep.fit_transform(train_X, train_y), timings, messages)
    assert issparse(matrix)
    available = prep.get_feature_names_out()
    selector = None
    selected = available
    if method:
        if method == "mi" and settings["k"] > len(available):
            raise ValueError(f"k={settings['k']} exceeds this fit's {len(available)} available features")
        selector = build_selector(method, **settings)
        matrix = timed_fit(
            "selection", lambda: selector.fit_transform(matrix, train_y), timings, messages
        )
        assert issparse(matrix)
        selected = selector.get_feature_names_out(available)
    timed_fit("classifier", lambda: model.fit(matrix, train_y), timings, messages)
    metrics = {
        "available": len(available), "selected": len(selected), **timings,
        "fit_s": sum(timings.values()), "kernel_peak_rss_mib": kernel_peak_rss_mib(),
        "convergence_warnings": sum("ConvergenceWarning:" in message for message in messages),
        "warnings": " | ".join(messages),
    }
    return prep, selector, model, available, selected, metrics

In [4]:
pilot_pool = folds[0][0]
accident_counts = pd.Series(groups[pilot_pool]).value_counts(sort=False)
shuffled = np.random.default_rng(RANDOM_STATE).permutation(len(accident_counts))
end = np.searchsorted(accident_counts.to_numpy()[shuffled].cumsum(), 100_000) + 1
pilot_accidents = accident_counts.index[shuffled[:end]]
pilot = pilot_pool[pd.Index(groups[pilot_pool]).isin(pilot_accidents)]
assert len(pilot) == accident_counts.loc[pilot_accidents].sum()
assert not set(groups[pilot]) & set(groups[folds[0][1]])
print(f"Pilot: {len(pilot):,} rows, {len(pilot_accidents):,} accidents, KSI {y[pilot].mean():.4%}")
print(f"Cumulative kernel peak RSS before pilot: {format_kernel_peak_rss()}")
pilot_records = []
for label, method, settings in [
    ("All features", None, {}),
    ("MI k=100", "mi", {"k": 100}),
    ("L1 C=0.05", "l1", {"C": 0.05}),
]:
    *_, diagnostics = fit_steps(X.iloc[pilot], y[pilot], method, **settings)
    pilot_records.append({"configuration": label, **diagnostics})
    del _
pilot_table = pd.DataFrame(pilot_records)
print(pilot_table.drop(columns="warnings").to_string(index=False, na_rep="unavailable", float_format="{:.2f}".format))

Pilot: 100,001 rows, 44,085 accidents, KSI 18.2848%
Cumulative kernel peak RSS before pilot: 1237.44 MiB
configuration  available  selected  selection_s  prep_s  classifier_s  fit_s  kernel_peak_rss_mib  convergence_warnings
 All features        388       388         0.00    0.96          0.67   1.63              1411.77                     0
     MI k=100        388       100         2.85    0.91          0.44   4.19              1480.03                     0
    L1 C=0.05        388       214        12.99    0.91          0.41  14.31              1623.09                     0


## Full-fold comparison

Fit each configuration independently and serially on all rows of its training fold. Inspect the fitted feature width before each MI selection. Fit times include training transformation, feature selection where applicable, and classifier fitting. They exclude validation prediction and scoring. Warnings and fit failures are reported alongside the results.

In [5]:
def run_configuration(label, method=None, **settings):
    oof = np.full(len(y), np.nan)
    records, feature_sets = [], []
    for fold, (train, valid) in enumerate(folds, start=1):
        start = perf_counter()
        try:
            prep, selector, model, available, selected, diagnostics = fit_steps(
                X.iloc[train], y[train], method, **settings
            )
            matrix = prep.transform(X.iloc[valid])
            if selector is not None:
                matrix = selector.transform(matrix)
            probabilities = model.predict_proba(matrix)[:, 1]
            del matrix
            oof[valid] = probabilities
            score = evaluate(y[valid], probabilities)["pr_auc"]
        except (ValueError, RuntimeError) as error:
            records.append({
                "configuration": label, "fold": fold, "failure": str(error),
                "elapsed_before_failure_s": perf_counter() - start,
            })
            print(f"{label}, fold {fold} failed: {error}", flush=True)
            continue
        records.append({
            "configuration": label, "fold": fold, "pr_auc": score,
            "retention": len(selected) / len(available), "failure": "", **diagnostics,
        })
        feature_sets.append((set(available), set(selected)))
        print(f"{label}, fold {fold}: AP={score:.6f}, "
              f"selected={len(selected)}/{len(available)}, fit={diagnostics['fit_s']:.2f} s",
              flush=True)
    pooled = evaluate(y, oof)["pr_auc"] if np.isfinite(oof).all() else np.nan
    return records, feature_sets, pooled


configurations = [
    ("All features", None, {}),
    *[(f"MI k={k}", "mi", {"k": k}) for k in (50, 100, 200)],
    *[(f"L1 C={c:g}", "l1", {"C": c}) for c in (0.005, 0.02, 0.05)],
]
results = {}

### All-features baseline

In [6]:
label, method, settings = configurations[0]
results[label] = run_configuration(label, method, **settings)

All features, fold 1: AP=0.588062, selected=394/394, fit=8.16 s
All features, fold 2: AP=0.589745, selected=394/394, fit=8.14 s
All features, fold 3: AP=0.588493, selected=394/394, fit=8.41 s
All features, fold 4: AP=0.588685, selected=394/394, fit=8.27 s
All features, fold 5: AP=0.589494, selected=394/394, fit=8.12 s


### Mutual information

In [7]:
for label, method, settings in configurations[1:4]:
    results[label] = run_configuration(label, method, **settings)

MI k=50, fold 1: AP=0.552386, selected=50/394, fit=22.50 s
MI k=50, fold 2: AP=0.556809, selected=50/394, fit=24.13 s
MI k=50, fold 3: AP=0.548323, selected=50/394, fit=23.89 s
MI k=50, fold 4: AP=0.551157, selected=50/394, fit=22.63 s
MI k=50, fold 5: AP=0.556788, selected=50/394, fit=23.08 s
MI k=100, fold 1: AP=0.576632, selected=100/394, fit=23.79 s
MI k=100, fold 2: AP=0.578064, selected=100/394, fit=24.50 s
MI k=100, fold 3: AP=0.575865, selected=100/394, fit=24.26 s
MI k=100, fold 4: AP=0.576574, selected=100/394, fit=24.57 s
MI k=100, fold 5: AP=0.577305, selected=100/394, fit=24.20 s
MI k=200, fold 1: AP=0.582865, selected=200/394, fit=24.37 s
MI k=200, fold 2: AP=0.588027, selected=200/394, fit=25.13 s
MI k=200, fold 3: AP=0.584715, selected=200/394, fit=25.80 s
MI k=200, fold 4: AP=0.584758, selected=200/394, fit=24.40 s
MI k=200, fold 5: AP=0.586082, selected=200/394, fit=24.67 s


### L1-regularized logistic regression

In [8]:
for label, method, settings in configurations[4:]:
    results[label] = run_configuration(label, method, **settings)

L1 C=0.005, fold 1: AP=0.581187, selected=137/394, fit=29.88 s
L1 C=0.005, fold 2: AP=0.583337, selected=139/394, fit=37.59 s
L1 C=0.005, fold 3: AP=0.582171, selected=136/394, fit=47.01 s
L1 C=0.005, fold 4: AP=0.583190, selected=139/394, fit=24.25 s
L1 C=0.005, fold 5: AP=0.582717, selected=141/394, fit=37.85 s
L1 C=0.02, fold 1: AP=0.587412, selected=253/394, fit=147.43 s
L1 C=0.02, fold 2: AP=0.589579, selected=249/394, fit=182.93 s
L1 C=0.02, fold 3: AP=0.588777, selected=253/394, fit=118.72 s
L1 C=0.02, fold 4: AP=0.587995, selected=246/394, fit=90.59 s
L1 C=0.02, fold 5: AP=0.588792, selected=251/394, fit=87.85 s
L1 C=0.05, fold 1: AP=0.587752, selected=300/394, fit=84.90 s
L1 C=0.05, fold 2: AP=0.589929, selected=293/394, fit=90.07 s
L1 C=0.05, fold 3: AP=0.588876, selected=299/394, fit=88.98 s
L1 C=0.05, fold 4: AP=0.588411, selected=298/394, fit=100.29 s
L1 C=0.05, fold 5: AP=0.588851, selected=295/394, fit=115.70 s


## Comparison and diagnostics

The mean and population standard deviation weight each fold equally. Pooled OOF average precision scores the concatenated out-of-fold probabilities. The historical value of 0.5887 in `results/baseline_train.csv` is a pooled score.

In [9]:
detail = pd.DataFrame([row for rows, _, _ in results.values() for row in rows])
print(detail.drop(columns=["warnings", "failure"]).to_string(index=False, na_rep="unavailable", float_format="{:.6f}".format))
issues = detail[detail["failure"].fillna("").ne("") | detail["warnings"].fillna("").ne("")]
if issues.empty:
    print("No warnings or failures in the full comparison.")
else:
    print(issues[["configuration", "fold", "warnings", "failure"]].to_string(index=False))

summary_rows = []
for label, (records, _, pooled) in results.items():
    completed = pd.DataFrame([row for row in records if not row["failure"]])
    row = {"configuration": label, "completed": len(completed)}
    if len(completed):
        row.update({
            "fold_mean_ap": completed["pr_auc"].mean(),
            "fold_sd_ap": completed["pr_auc"].std(ddof=0),
            "pooled_oof_ap": pooled,
            "mean_available": completed["available"].mean(),
            "mean_selected": completed["selected"].mean(),
            "mean_retention": completed["retention"].mean(),
            "total_fit_s": completed["fit_s"].sum(),
            "convergence_warnings": completed["convergence_warnings"].sum(),
        })
    summary_rows.append(row)
summary = pd.DataFrame(summary_rows).set_index("configuration")
print(summary.to_string(float_format="{:.6f}".format))

configuration  fold   pr_auc  retention  available  selected  selection_s   prep_s  classifier_s      fit_s  kernel_peak_rss_mib  convergence_warnings
 All features     1 0.588062   1.000000        394       394     0.000000 4.818962      3.338964   8.157925          1713.203125                     0
 All features     2 0.589745   1.000000        394       394     0.000000 4.598096      3.540090   8.138185          1844.437500                     0
 All features     3 0.588493   1.000000        394       394     0.000000 4.702111      3.707926   8.410037          1844.437500                     0
 All features     4 0.588685   1.000000        394       394     0.000000 4.613262      3.656747   8.270009          1866.093750                     0
 All features     5 0.589494   1.000000        394       394     0.000000 4.650140      3.474842   8.124983          1866.093750                     0
      MI k=50     1 0.552386   0.126904        394        50    17.018157 4.607900      0.8744

## Feature stability

Availability counts the folds in which the encoder produced a column. Retention counts the folds in which the fitted selector kept it. Entries are shown as retained/available. An infrequent-category column can represent different constituent levels in different folds.

In [10]:
all_names = sorted(set().union(*(
    available for _, sets, _ in results.values() for available, _ in sets
)))
stability = pd.DataFrame(index=pd.Index(all_names, name="transformed_feature"))
for label, (_, sets, _) in results.items():
    available_counts = pd.Series({name: sum(name in avail for avail, _ in sets) for name in all_names})
    retained_counts = pd.Series({name: sum(name in kept for _, kept in sets) for name in all_names})
    stability[label] = retained_counts.astype(str) + "/" + available_counts.astype(str)
    if label != "All features":
        always = retained_counts.index[retained_counts == len(folds)].tolist()
        print(f"{label}: {len(always)} features retained in all five folds")
selector_columns = stability.drop(columns="All features")
common = selector_columns.index[selector_columns.eq("5/5").all(axis=1)].tolist()
print(f"Retained by all six selector configurations in all five folds ({len(common)}):")
print(", ".join(common))
print(stability.to_string())

MI k=50: 49 features retained in all five folds
MI k=100: 98 features retained in all five folds
MI k=200: 193 features retained in all five folds
L1 C=0.005: 128 features retained in all five folds
L1 C=0.02: 216 features retained in all five folds
L1 C=0.05: 256 features retained in all five folds
Retained by all six selector configurations in all five folds (39):
age, agg_1, belt_or_helmet_recorded, catr_3, catr_4, catu_3, choc_1, choc_4, circ_1, circ_2, col_2, col_4, col_6, col_7, dep_75, dep_94, etatp_1, hour, locp_0, locp_None, lum_3, manv_13, nbv, obs_0, obs_2, obsm_0, obsm_2, place_10, road_user_Car, road_user_Motorcycle >50cc, road_user_Pedestrian, secu1_1, secu1_2, secu2_None, situ_1, situ_3, trajet_4, trajet_5, vma
                                         All features MI k=50 MI k=100 MI k=200 L1 C=0.005 L1 C=0.02 L1 C=0.05
transformed_feature                                                                                           
actp_1                                    

## Interpretation

- **Recommended subset.** L1 at C=0.02 retained 250.4 of 394 columns on average, a 36.4% reduction. Its mean PR-AUC of 0.588511 was 0.000385 below the all-features mean. C=0.05 recovered 0.000253 at the cost of 46.6 additional columns on average. Keep the full baseline as the reference when feature count is unconstrained.
- **Filter trade-off.** MI at k=200 achieved 0.585289 mean PR-AUC with 200 columns and about one quarter of the five-fold fit time of L1 at C=0.02. Reducing k further came with larger PR-AUC drops.
- **Stability.** L1 at C=0.02 retained 216 columns in all five folds. Across all six selectors, 39 columns were always retained, including age, hour, speed limit (vma), the recorded belt-or-helmet flag, and several road-user categories. This is stability across training folds, with no causal interpretation.
- **Cost.** All 35 fits ran serially with one thread requested for native numerical libraries. The comparison table reports fit times, excluding validation and fold setup. Full-fold MI required no subsampling. Reported cumulative kernel peak RSS includes data loading and all preceding fits. There were no model-fit warnings or failures. PyArrow emitted sandbox diagnostics when querying CPU capabilities and completed the filtered read.
- **Scope.** The same training CV folds were used to compare and choose configurations. The observed differences are descriptive and performance in the held-out years remains unmeasured.

## Data leakage demonstrations

The deliberately incorrect procedures below are educational demonstrations only. Their scores are excluded from model selection and the Phase 3 recommendation. All comparisons use the same training-year sample and the shared average-precision metric. Wrong minus right denotes the difference in mean fold PR-AUC.

In [11]:
from scipy.sparse import vstack
from sklearn.model_selection import KFold

demo_start = perf_counter()
demo_counts = pd.Series(groups).value_counts(sort=False)
demo_order = np.random.default_rng(RANDOM_STATE).permutation(len(demo_counts))
demo_end = np.searchsorted(demo_counts.to_numpy()[demo_order].cumsum(), 100_000) + 1
demo_accidents = demo_counts.index[demo_order[:demo_end]]
demo_indices = np.flatnonzero(pd.Index(groups).isin(demo_accidents))
demo_X = X.iloc[demo_indices].reset_index(drop=True)
demo_y, demo_groups = y[demo_indices], groups[demo_indices]
assert len(demo_X) == demo_counts.loc[demo_accidents].sum()
assert GROUP_COL not in demo_X.columns

demo_grouped = list(get_cv().split(demo_X, demo_y, demo_groups))
demo_rowwise = list(KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE).split(demo_X))
print(f"Sample: {len(demo_X):,} people, {len(demo_accidents):,} accidents, "
      f"{demo_y.sum():,} KSI cases ({demo_y.mean():.4%})")
print(f"Sample preparation and split generation: {perf_counter() - demo_start:.2f} s")

demo_warnings = []
demo_results = {}

def run_demo(splits, predict_fold, setup_fit_s=0.0):
    oof_demo = np.full(len(demo_y), np.nan)
    hits = np.zeros(len(demo_y), dtype=np.int8)
    rows = []
    for fold, (train, valid) in enumerate(splits, start=1):
        probabilities, fitted_seconds = predict_fold(train, valid)
        assert np.isfinite(probabilities).all()
        oof_demo[valid] = probabilities
        hits[valid] += 1
        train_accidents = set(demo_groups[train])
        valid_accidents = set(demo_groups[valid])
        shared = train_accidents & valid_accidents
        shared_rows = sum(accident in shared for accident in demo_groups[valid])
        rows.append({
            "fold": fold,
            "pr_auc": evaluate(demo_y[valid], probabilities)["pr_auc"],
            "fit_s": fitted_seconds,
            "train_ksi_rate": demo_y[train].mean(),
            "validation_ksi_rate": demo_y[valid].mean(),
            "shared_accidents": len(shared),
            "validation_accidents": len(valid_accidents),
            "shared_validation_rows": shared_rows,
            "validation_rows": len(valid),
            "shared_validation_share": shared_rows / len(valid),
        })
    assert (hits == 1).all()
    return pd.DataFrame(rows), evaluate(demo_y, oof_demo)["pr_auc"], setup_fit_s


def honest_logistic_fold(train, valid):
    pipeline = logistic_baseline()
    times = {}
    timed_fit("fold_pipeline", lambda: pipeline.fit(demo_X.iloc[train], demo_y[train]),
              times, demo_warnings)
    return pipeline.predict_proba(demo_X.iloc[valid])[:, 1], sum(times.values())

Sample: 100,002 people, 44,175 accidents, 18,243 KSI cases (18.2426%)
Sample preparation and split generation: 2.68 s


### A. Preprocessing leakage

**Wrong:** fit the baseline preprocessor on the full sample before grouped CV. Validation rows then influence imputation medians, scaling statistics, and which one-hot levels count as infrequent. KSI labels are absent from that preprocessing fit.

**Right:** refit the all-features logistic pipeline on each training fold. The grouped validation indices and classifier settings match the wrong version.

In [12]:
global_prep = logistic_baseline().named_steps["prep"]
global_times = {}
globally_transformed = timed_fit(
    "global_prep", lambda: global_prep.fit_transform(demo_X), global_times, demo_warnings
)
assert issparse(globally_transformed)

def globally_preprocessed_fold(train, valid):
    model = logistic_baseline().named_steps["model"]
    times = {}
    timed_fit("classifier", lambda: model.fit(globally_transformed[train], demo_y[train]),
              times, demo_warnings)
    return model.predict_proba(globally_transformed[valid])[:, 1], sum(times.values())

demo_results[("A", "Wrong")] = run_demo(
    demo_grouped, globally_preprocessed_fold, setup_fit_s=sum(global_times.values())
)
del globally_transformed, global_prep
demo_results[("A", "Right")] = run_demo(demo_grouped, honest_logistic_fold)
assert demo_results[("A", "Right")][0]["shared_accidents"].eq(0).all()
print(f"Wrong A one-time global preprocessing fit: {sum(global_times.values()):.2f} s")

Wrong A one-time global preprocessing fit: 0.92 s


### B. Feature-selection leakage

Both versions fit the same preprocessor on the training fold, use the same transformed columns and select 100 features with MI. Both final classifiers fit on the training fold.

**Wrong:** score MI using the transformed training and validation rows and their KSI labels. **Right:** score MI using only the transformed training rows and their labels. This isolates label leakage during feature selection. The guide's select-before-CV example has the same problem because each validation label can influence the selected columns. Here the representation is fitted separately within each fold so that preprocessing leakage does not confound the comparison.

In [13]:
mi_k = 100
paired_mi = {"Wrong": {}, "Right": {}}
for train, valid in demo_grouped:
    prep = MutualInformationPreprocessor([("columns", logistic_baseline().named_steps["prep"])])
    prep_times = {}
    train_matrix = timed_fit("mi_prep", lambda: prep.fit_transform(demo_X.iloc[train], demo_y[train]),
                             prep_times, demo_warnings)
    valid_matrix = prep.transform(demo_X.iloc[valid])
    assert issparse(train_matrix) and issparse(valid_matrix)
    assert mi_k <= train_matrix.shape[1] == valid_matrix.shape[1]

    for variant in ("Wrong", "Right"):
        times = dict(prep_times)
        selector = build_selector("mi", k=mi_k)
        scoring_matrix, scoring_labels = train_matrix, demo_y[train]
        if variant == "Wrong":
            # Educational only: deliberately expose validation labels to MI.
            scoring_matrix = vstack((train_matrix, valid_matrix), format="csr")
            scoring_matrix.mi_discrete_mask = train_matrix.mi_discrete_mask
            scoring_labels = demo_y[np.concatenate((train, valid))]
        timed_fit("mi_selection", lambda: selector.fit(scoring_matrix, scoring_labels),
                  times, demo_warnings)
        del scoring_matrix, scoring_labels
        assert selector.get_support().sum() == mi_k
        selected_train, selected_valid = selector.transform(train_matrix), selector.transform(valid_matrix)
        model = logistic_baseline().named_steps["model"]
        try:
            timed_fit("classifier", lambda: model.fit(selected_train, demo_y[train]), times, demo_warnings)
            predicted = model.predict_proba(selected_valid)[:, 1]
        finally:
            del selected_train, selected_valid
        paired_mi[variant][valid[0]] = predicted, sum(times.values())
    del train_matrix, valid_matrix

for variant, predictions in paired_mi.items():
    demo_results[("B", variant)] = run_demo(
        demo_grouped, lambda train, valid: predictions[valid[0]]
    )
print(f"Both MI variants kept k={mi_k} in all five folds. "
      "The shared fold-preprocessing cost is attributed to each variant.")

Both MI variants kept k=100 in all five folds. The shared fold-preprocessing cost is attributed to each variant.


### C. Accident-group leakage

**Wrong:** use shuffled row-level KFold. **Right:** group people by accident using the already-computed grouped folds. Both versions fit the same all-features logistic pipeline within each fold. The grouped fits are identical to A-right, so their predictions and timings are reused.

People in the same accident share conditions such as the crash location, impact, and road environment. A row-level split can train on a person from the same accident as someone in validation, so validation can overstate performance on new accidents. The splitters also produce different fold compositions and KSI prevalence, both of which can influence measured average precision.

In [14]:
demo_results[("C", "Wrong")] = run_demo(demo_rowwise, honest_logistic_fold)
demo_results[("C", "Right")] = demo_results[("A", "Right")]
overlap_columns = [
    "fold", "shared_accidents", "validation_accidents", "shared_validation_rows",
    "validation_rows", "shared_validation_share", "train_ksi_rate", "validation_ksi_rate", "pr_auc",
]
for variant in ("Wrong", "Right"):
    overlap_table = demo_results[("C", variant)][0][overlap_columns]
    print(f"C {variant}")
    print(overlap_table.to_string(index=False, float_format="{:.6f}".format))

C Wrong
 fold  shared_accidents  validation_accidents  shared_validation_rows  validation_rows  shared_validation_share  train_ksi_rate  validation_ksi_rate   pr_auc
    1             14083                 16790                   16139            20001                 0.806910        0.181998             0.184141 0.601961
    2             14193                 16810                   16296            20001                 0.814759        0.182273             0.183041 0.583498
    3             14147                 16757                   16257            20000                 0.812850        0.182458             0.182300 0.576525
    4             14108                 16761                   16237            20000                 0.811850        0.183145             0.179550 0.585922
    5             14112                 16798                   16207            20000                 0.810350        0.182258             0.183100 0.588392
C Right
 fold  shared_accidents  validation_

### Comparison

The table gives the PR-AUC for each of the five folds, their unweighted mean and population standard deviation, and pooled out-of-fold PR-AUC. Fit time includes model fitting and the preprocessing and selection fits applicable to that variant. The one-time preprocessing fit in A-wrong is included in its total. A-right and C-right share the same fit time. Prediction and scoring time are excluded from fit totals.

In [15]:
comparison_rows = []
for (demo, variant), (frame, pooled, setup_fit_s) in demo_results.items():
    scores = frame["pr_auc"].to_numpy()
    comparison_rows.append({
        "demo": demo, "variant": variant,
        **{f"fold_{fold}": score for fold, score in enumerate(scores, start=1)},
        "mean": scores.mean(), "population_sd": scores.std(ddof=0), "pooled_oof": pooled,
        "total_fit_s": frame["fit_s"].sum() + setup_fit_s,
    })
demo_comparison = pd.DataFrame(comparison_rows).set_index(["demo", "variant"])
print(demo_comparison.to_string(float_format="{:.6f}".format))
demo_means = demo_comparison["mean"].unstack()
demo_deltas = demo_means["Wrong"] - demo_means["Right"]
print("\nWrong minus right mean PR-AUC")
print(demo_deltas.to_string(float_format="{:+.6f}".format))
demo_fit_times = pd.DataFrame({
    f"{demo} {variant}": frame["fit_s"].to_numpy()
    for (demo, variant), (frame, _, _) in demo_results.items()
}, index=pd.Index(range(1, 6), name="fold"))
print("\nPer-fold fit time in seconds")
print(demo_fit_times.to_string(float_format="{:.2f}".format))
print(f"Demonstration elapsed time: {perf_counter() - demo_start:.2f} s")
print(f"Cumulative kernel peak RSS through leakage demonstrations: {format_kernel_peak_rss()}")
print(f"Fit warnings: {len(demo_warnings)}")

               fold_1   fold_2   fold_3   fold_4   fold_5     mean  population_sd  pooled_oof  total_fit_s
demo variant                                                                                              
A    Wrong   0.587256 0.582009 0.578941 0.593859 0.585280 0.585469       0.005064    0.584976     3.789744
     Right   0.587793 0.582295 0.578595 0.593754 0.585421 0.585571       0.005125    0.585053     6.670721
B    Wrong   0.578123 0.572186 0.575462 0.576957 0.575433 0.575632       0.001995    0.575366    20.182867
     Right   0.577544 0.572075 0.575875 0.576839 0.574599 0.575386       0.001928    0.575133    16.911995
C    Wrong   0.601961 0.583498 0.576525 0.585922 0.588392 0.587259       0.008350    0.586951     6.671328
     Right   0.587793 0.582295 0.578595 0.593754 0.585421 0.585571       0.005125    0.585053     6.670721

Wrong minus right mean PR-AUC
demo
A   -0.000102
B   +0.000246
C   +0.001688

Per-fold fit time in seconds
      A Wrong  A Right  B Wrong  B R

### Interpretation of the measured differences

- **A · Preprocessing:** The wrong mean was 0.000102 lower. Learning preprocessing statistics from validation features changed the evaluation slightly on this sample. The unsupervised leakage did not produce an upward shift here.
- **B · Selection:** The wrong mean was 0.000246 higher. The validation labels influenced the feature ranking, which makes the validation score unsuitable for model selection even with this small gap.
- **C · Grouping:** The wrong mean was 0.001688 higher. Row-level CV exposed 81,136 of 100,002 validation predictions (81.13%) to training rows from the same accident. Grouped CV had zero overlap. Validation KSI prevalence ranged from 17.96% to 18.41% with row-level CV and stayed near 18.24% with grouped CV. These different fold compositions prevent attributing the entire score gap to accident overlap.

The six variants are descriptive comparisons on one fixed sample. These demonstrations leave the Phase 3 recommendation at L1 with C=0.02.

## Methodology decisions

| Decision | Rationale and limitations |
|---|---|
| Train and validate on 2019–2023 | Apply the Parquet year filter before materializing the training frame. Reserve 2024–2025 for final evaluation. Training CV does not measure temporal generalization. |
| Use five stratified, accident-grouped folds | Group by `Num_Acc` to keep people from the same accident together. Reuse the same indices across configurations. Stratification aims to balance KSI prevalence across folds. |
| Fit preprocessing within each training fold | Learn imputation, scaling, and category pooling exclusively from training rows. The encoded width and available names can vary by fold. |
| Use mutual information as the filter | Rank each transformed column by its dependence on KSI. Univariate ranking can miss joint effects and retain redundant columns. |
| Use L1 logistic regression as the embedded selector | Select by coefficient magnitude from a jointly fitted sparse-input model. Correlated predictors can substitute for one another across folds. |
| Separate continuous and discrete MI features | Score one-hot features, binary flags, and missingness indicators as discrete. Score ordered numeric magnitudes with the continuous MI estimator, converting one column at a time to dense. Ties in numeric inputs make the estimate sensitive to the estimator's seeded jitter. |
| Use PR-AUC as the primary metric | Average precision emphasizes ranking of the less common KSI class and depends on prevalence. Report the unweighted fold mean, population SD, and pooled out-of-fold score. |
| Recommend L1 at C=0.02 for feature reduction | The mean count fell from 394 to 250.4 (36.4%), with an absolute mean PR-AUC decrease of 0.000385. This is a descriptive training-CV trade-off. Neither retention nor PR-AUC estimates a causal effect or held-out performance. |
| Sample whole accidents for leakage demonstrations | Use the project seed and keep all members of sampled accidents, preserving the dependence that row-level CV can expose. The observed gaps describe this fixed sample. |

## Report figures

The plots use the comparison and overlap tables computed above. Error bars show one population standard deviation across the five folds. The leakage figure separates score differences from the share of validation rows with an accident represented in training.

In [16]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from src.config import FIGURES_DIR

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(8, 4.8), layout="constrained")
fig.suptitle("Feature retention and validation PR-AUC", fontsize=16, fontweight="bold")
ax.set_title("Descriptive comparison across five grouped training folds", fontsize=10, loc="left")

families = {
    "Baseline": (["All features"], "#475569", "D"),
    "Mutual information": ([name for name in summary.index if name.startswith("MI ")], "#176B98", "o"),
    "L1 logistic regression": ([name for name in summary.index if name.startswith("L1 ")], "#A0421A", "s"),
}
label_positions = {
    "All features": (-4, -19, "right"),
    "MI k=50": (10, 3, "left"),
    "MI k=100": (0, -18, "center"),
    "MI k=200": (-4, 12, "right"),
    "L1 C=0.005": (0, 11, "center"),
    "L1 C=0.02": (0, 11, "center"),
    "L1 C=0.05": (7, -19, "left"),
}
for family, (names, color, marker) in families.items():
    values = summary.loc[names]
    ax.errorbar(values["mean_selected"], values["fold_mean_ap"], yerr=values["fold_sd_ap"],
                fmt=marker, color=color, capsize=3, linewidth=1.35, markersize=6, label=family)
    for name, row in values.iterrows():
        dx, dy, align = label_positions[name]
        recommended = name == "L1 C=0.02"
        label = "L1 C=0.02 • recommended" if recommended else name
        ax.annotate(label, (row["mean_selected"], row["fold_mean_ap"]),
                    xytext=(dx, dy), textcoords="offset points", ha=align, fontsize=9,
                    color=color, fontweight="bold" if recommended else "normal")

chosen = summary.loc["L1 C=0.02"]
ax.scatter(chosen["mean_selected"], chosen["fold_mean_ap"], s=155, marker="s",
           facecolors="none", edgecolors="#A0421A", linewidths=1.4, zorder=4)
ax.set(xlim=(25, 425), ylim=(0.545, 0.601), xlabel="Mean number of retained features",
       ylabel="Mean validation PR-AUC")
ax.set_yticks([0.55, 0.56, 0.57, 0.58, 0.59, 0.60])
ax.grid(axis="y", color="#E2E8F0")
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(loc="lower right", title="Bars show ±1 fold SD", fontsize=9, frameon=False)
feature_path = FIGURES_DIR / "m2_feature_selection.png"
fig.savefig(feature_path, dpi=300, facecolor="white")
plt.close(fig)
print(f"Saved {feature_path.name}")

Matplotlib is building the font cache; this may take a moment.


Saved m2_feature_selection.png


In [17]:
fig, (score_ax, overlap_ax) = plt.subplots(1, 2, figsize=(10, 4.3), layout="constrained")
fig.suptitle("Leakage demonstrations on the training sample", fontsize=16, fontweight="bold")

difference_pp = 100 * demo_deltas.loc[["A", "B", "C"]]
demo_labels = ["A  Preprocessing", "B  Feature selection", "C  Accident grouping"]
score_ax.barh(demo_labels, difference_pp, color="#176B98", height=0.48)
score_ax.axvline(0, color="#475569", linewidth=1)
score_ax.invert_yaxis()
# Use a symmetric scale with a minimum span of 0.6 percentage points.
score_limit = max(0.30, 1.4 * difference_pp.abs().max())
score_ax.set(xlim=(-score_limit, score_limit), xlabel="Wrong − right mean PR-AUC (percentage points)")
score_ax.set_title("Score difference", loc="left", fontweight="bold")
for position, value in enumerate(difference_pp):
    offset = 5 if value >= 0 else -5
    score_ax.annotate(f"{value:+.4f}", (value, position), xytext=(offset, 0),
                      textcoords="offset points", ha="left" if value >= 0 else "right",
                      va="center", fontsize=9)

overlap_rates = []
for variant in ("Wrong", "Right"):
    fold_overlap = demo_results[("C", variant)][0]
    overlap_rates.append(fold_overlap["shared_validation_rows"].sum() /
                         fold_overlap["validation_rows"].sum())
overlap_ax.barh(["Row-level KFold", "Grouped CV"], overlap_rates, color="#A0421A", height=0.48)
overlap_ax.invert_yaxis()
overlap_ax.set(xlim=(0, 1), xticks=[0, 0.25, 0.5, 0.75, 1],
               xlabel="Validation rows with an accident in training")
overlap_ax.xaxis.set_major_formatter(PercentFormatter(1))
overlap_ax.set_title("Accident overlap", loc="left", fontweight="bold")
for position, value in enumerate(overlap_rates):
    overlap_ax.annotate(f"{value:.2%}", (value, position), xytext=(5, 0),
                        textcoords="offset points", va="center", fontsize=9)

for axis in (score_ax, overlap_ax):
    axis.set_axisbelow(True)
    axis.grid(axis="x", color="#E2E8F0")
    axis.spines[["top", "right", "left"]].set_visible(False)
    axis.tick_params(axis="y", length=0)
leakage_path = FIGURES_DIR / "m2_leakage_demo.png"
fig.savefig(leakage_path, dpi=300, facecolor="white")
plt.close(fig)
print(f"Saved {leakage_path.name}")

Saved m2_leakage_demo.png
